In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset, load_dataset
import torch
from torch.utils.data import DataLoader
import seaborn as sns
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
tokenized_imdb = dataset.map(lambda x: tokenizer(x["text"], padding="max_length", truncation=True), batched=True)
from transformers import pipeline


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [9]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [2]:
tokenized_imdb = tokenized_imdb.remove_columns(["text"])
tokenized_imdb = tokenized_imdb.rename_column("label", "labels")
tokenized_imdb.set_format("torch")
train_dataset = tokenized_imdb["train"]
test_dataset = tokenized_imdb["test"]
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)
eval_dataloader = DataLoader(test_dataset, batch_size=8)
tokenized_imdb['train']
tokenized_imdb['test']


Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})

In [21]:
from transformers import TrainingArguments, Trainer

# 1. Задаем настройки обучения
training_args = TrainingArguments(
    output_dir="/results",          # Папка, куда будут сохраняться веса модели
    eval_strategy="epoch",           # Проверять качество на тестовой выборке каждую эпоху
    learning_rate=2e-5,              # Стандартная скорость обучения для тонкой настройки BERT
    per_device_train_batch_size=8,   # Размер батча для обучения
    per_device_eval_batch_size=8,    # Размер батча для оценки
    num_train_epochs=3,              # Количество полных проходов по данным (эпох)
    weight_decay=0.01,               # Тот самый параметр регуляризации для AdamW
)

# 2. Инициализируем Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"], # Передаем датасеты напрямую
    eval_dataset=tokenized_imdb["test"],
)

# 3. Запускаем магию!
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.256552,0.334801
2,0.143176,0.277293
3,0.087753,0.315847


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=0.1765352996826172, metrics={'train_runtime': 1924.5985, 'train_samples_per_second': 38.969, 'train_steps_per_second': 4.871, 'total_flos': 1.9733329152e+16, 'train_loss': 0.1765352996826172, 'epoch': 3.0})

In [18]:
save_directory = "/saved_bert_model"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

NameError: name 'trainer' is not defined

In [3]:
os.chdir(r"C:\Users\egorm\ML_ENV\neural_2_sem")
local_model_path = "../local_distilbert"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model_distil = AutoModelForSequenceClassification.from_pretrained(local_model_path, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: ../local_distilbert
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
model_distil.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [6]:
# 1. Задаем настройки обучения
training_args = TrainingArguments(
    output_dir="results_distil",          # Папка, куда будут сохраняться веса модели
    eval_strategy="epoch",           # Проверять качество на тестовой выборке каждую эпоху
    learning_rate=2e-5,              # Стандартная скорость обучения для тонкой настройки BERT
    per_device_train_batch_size=8,   # Размер батча для обучения
    per_device_eval_batch_size=8,    # Размер батча для оценки
    num_train_epochs=3,              # Количество полных проходов по данным (эпох)
    weight_decay=0.01,               # Тот самый параметр регуляризации для AdamW
)

In [7]:
# 2. Инициализируем Trainer
trainer = Trainer(
    model=model_distil,
    args=training_args,
    train_dataset=tokenized_imdb["train"], # Передаем датасеты напрямую
    eval_dataset=tokenized_imdb["test"],
)

In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.267780,0.326955
2,0.168270,0.288376
3,0.093049,0.332675


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=0.1901795037841797, metrics={'train_runtime': 1072.6073, 'train_samples_per_second': 69.923, 'train_steps_per_second': 8.74, 'total_flos': 9935054899200000.0, 'train_loss': 0.1901795037841797, 'epoch': 3.0})

In [9]:
# Указываем название папки, куда всё сохранится
save_directory = "saved_distilbert_model"

# 1. Сохраняем веса обученной модели
trainer.save_model(save_directory)

# 2. Обязательно сохраняем токенизатор туда же!
tokenizer.save_pretrained(save_directory)

print(f"Успех! Модель и токенизатор надежно сохранены в папку: {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Успех! Модель и токенизатор надежно сохранены в папку: saved_distilbert_model


In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\egorm\miniconda3\envs\ml_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\egorm\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
model_distil = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
from transformers import pipeline

# 1. Прописываем пути к нашим сохраненным моделям
bert_path = "../saved_bert_model"             # Путь к обученному BERT
distilbert_path = "../saved_distilbert_model" # Путь к обученному DistilBERT

print("Загружаем модели с диска...")

# 2. Создаем два независимых пайплайна
bert_analyzer = pipeline("text-classification", model=bert_path, tokenizer=bert_path)
distilbert_analyzer = pipeline("text-classification", model=distilbert_path, tokenizer=distilbert_path)

# 3. Пишем тестовые отзывы на английском (один хороший, один плохой)
test_reviews = [
    "This movie was an absolute masterpiece! The acting was brilliant and the plot kept me engaged until the very end.",
    "Terrible film. I completely wasted two hours of my life, the characters were so flat and boring."
]

print("✅ Модели загружены! Начинаем сравнение:\n")
print("=" * 60)

# 4. Прогоняем тексты через обе нейросети
for review in test_reviews:
    print(f"📝 Отзыв: {review}\n")

    # Получаем предсказания (LABEL_1 обычно позитив, LABEL_0 - негатив)
    bert_result = bert_analyzer(review)[0]
    distil_result = distilbert_analyzer(review)[0]

    print(f"🤖 BERT:       {bert_result['label']} (уверенность: {bert_result['score']:.4f})")
    print(f"⚡ DistilBERT: {distil_result['label']} (уверенность: {distil_result['score']:.4f})")
    print("=" * 60)